# Speech enhancement with USES

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/espnet/notebook/blob/master/Demos/enh_demo.ipynb) [![enh_demo](https://github.com/espnet/notebook/actions/workflows/enh_demo.yml/badge.svg)](https://github.com/espnet/notebook/actions/workflows/enh_demo.yml)

One model for noise, reverberation, one microphone or several, and any
sample rate. Noisy speech in, clean speech out.

CPU works and is slow — a few times real time. The model is 13 MB.


## Install

`[enh]` brings `fast_bss_eval`, which the enhancement losses import when
the model is built.


In [ ]:
%pip install -q "espnet[enh]==202610.post1" espnet_model_zoo librosa


## Something noisy

Clean speech with noise added, so there is a clean reference to compare
against afterwards.


In [ ]:
import numpy as np, librosa, soundfile as sf
from IPython.display import Audio, display

!wget -q -O clean.wav https://github.com/espnet/espnet/raw/master/test_utils/ctc_align_test.wav
clean, rate = librosa.load("clean.wav", sr=16000)
noisy = clean + 0.05 * np.random.default_rng(0).standard_normal(len(clean))
sf.write("noisy.wav", noisy.astype("float32"), rate)

print("noisy")
display(Audio(noisy, rate=rate))


## Enhance it

[`USES`](https://huggingface.co/espnet/Wangyou_Zhang_universal_train_enh_uses_refch0_2mem_raw)
takes `(batch, samples)` and the rate the audio is at — it is the same model
whether that is 8 kHz or 48 kHz. It returns one wave per output stream: one
here, one per speaker for a separation model.


In [ ]:
from espnet2.bin.enh_inference import SeparateSpeech

enh = SeparateSpeech.from_pretrained(
    "espnet/Wangyou_Zhang_universal_train_enh_uses_refch0_2mem_raw", device="cpu"
)

waves = enh(noisy[None, ...].astype("float32"), fs=rate)
print(f"{len(waves)} output stream(s)")
display(Audio(waves[0][0], rate=rate))


## Did it help

Signal-to-noise against the clean reference, before and after. Higher is
better; the number is in decibels.


In [ ]:
def snr(reference, estimate):
    n = min(len(reference), len(estimate))
    reference, estimate = reference[:n], estimate[:n]
    scale = float(reference @ estimate / (reference @ reference + 1e-9))
    error = estimate - scale * reference
    return 10 * np.log10(float(scale**2 * (reference @ reference)) /
                         float(error @ error + 1e-12) + 1e-12)

print(f"noisy    {snr(clean, noisy):6.2f} dB")
print(f"enhanced {snr(clean, waves[0][0]):6.2f} dB")


## Where next

- **From the terminal**: `espnet enhance noisy.wav -o clean.wav`
- **In the browser**: the [universal-se Space](https://huggingface.co/spaces/espnet/universal-se), which reads the rates
  this model was trained on off the checkpoint
- **Separation** — two people talking at once — and scoring with VERSA:
  [`../Courses/CMUSpeechTechnology26S/speech_enhancement.ipynb`](../Courses/CMUSpeechTechnology26S/speech_enhancement.ipynb)
